# 07 — Locked Continuous Evaluation

This notebook evaluates the predictive distributions generated and
locked in Notebook 06.

The raw deterministic forecast, the selected uncalibrated distribution
and the selected calibrated distribution are compared separately on
the holdout and June external blocks.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "continuous_evaluation_spec.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/"
            "evaluate_locked_continuous_predictions.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Locked continuous evaluation failed."
    )


 LOCKED CONTINUOUS EVALUATION COMPLETE

Selected model: pooled_empirical_residual
Selected family: empirical_residual
Locked dispersion scale: 1.25

Evaluation rows per variant: 159
Evaluation dates: 40
Holdout dates: 10
External-test dates: 30

Primary block summary:
chronology_block      forecast_variant  mean_date_crps  standard_error_date_crps  relative_crps_reduction_vs_raw  median_mae_c  median_bias_c  empirical_50_coverage  empirical_80_coverage  empirical_90_coverage  mean_80_width_c
         holdout     raw_deterministic        1.902500                  0.216905                        0.000000      1.902500      -1.877500               0.000000                0.00000                0.00000            0.000
         holdout selected_uncalibrated        0.470421                  0.072149                        0.752735      0.632500      -0.177500               0.675000                0.95000                0.95000            2.900
         holdout   selected_calibrated        

## Primary score

The continuous ranked probability score evaluates the full predictive
distribution.

A score is first calculated for every date and decision-rule
observation. Scores are then averaged within settlement date, after
which the date-level values are averaged within each evaluation block.

Settlement date is therefore the primary uncertainty unit.

In [2]:
summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "07_continuous_block_summary.csv"
)

print(
    summary[
        [
            "chronology_block",
            "forecast_variant_display",
            "mean_date_crps",
            "standard_error_date_crps",
            "relative_crps_reduction_vs_raw",
            "median_mae_c",
            "median_bias_c",
            "empirical_80_coverage",
            "mean_80_width_c",
        ]
    ].to_string(index=False)
)

chronology_block forecast_variant_display  mean_date_crps  standard_error_date_crps  relative_crps_reduction_vs_raw  median_mae_c  median_bias_c  empirical_80_coverage  mean_80_width_c
         holdout        Raw deterministic        1.902500                  0.216905                        0.000000      1.902500      -1.877500                0.00000            0.000
         holdout    Selected uncalibrated        0.470421                  0.072149                        0.752735      0.632500      -0.177500                0.95000            2.900
         holdout      Selected calibrated        0.498926                  0.061829                        0.737752      0.632500      -0.177500                0.95000            3.625
   external_test        Raw deterministic        1.780833                  0.152615                        0.000000      1.783193      -1.719328                0.00000            0.000
   external_test    Selected uncalibrated        0.577711                  

## Secondary diagnostics

Predictive median error measures the central location of each
distribution.

Empirical coverage and average width are reported for the central
50 per cent, 80 per cent and 90 per cent intervals. Coverage measures
reliability, whereas interval width measures concentration.

These diagnostics did not determine model or calibration selection.

In [3]:
pairwise = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "07_continuous_pairwise_comparison.csv"
)

print(
    pairwise[
        [
            "chronology_block",
            "left_display",
            "right_display",
            "paired_dates",
            "mean_paired_crps_difference",
            "descriptive_95_interval_lower",
            "descriptive_95_interval_upper",
            "left_better_date_share",
        ]
    ].to_string(index=False)
)

chronology_block          left_display         right_display  paired_dates  mean_paired_crps_difference  descriptive_95_interval_lower  descriptive_95_interval_upper  left_better_date_share
         holdout Selected uncalibrated     Raw deterministic            10                    -1.432079                      -1.965917                      -0.898240                     0.9
         holdout   Selected calibrated     Raw deterministic            10                    -1.403574                      -1.932710                      -0.874437                     0.9
         holdout   Selected calibrated Selected uncalibrated            10                     0.028505                       0.004452                       0.052558                     0.3
   external_test Selected uncalibrated     Raw deterministic            30                    -1.203122                      -1.506643                      -0.899601                     0.9
   external_test   Selected calibrated     Raw det

## Paired differences

Each comparison uses date-level CRPS differences. A negative
difference means that the forecast listed on the left has the lower
CRPS.

The reported intervals use the standard error across settlement
dates. They are descriptive rather than formal significance tests
because dates may be serially dependent and the holdout contains only
ten dates.

In [4]:
manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "07_continuous_evaluation_manifest.json"
    ).read_text(encoding="utf-8")
)

assert manifest["evaluation_locked"] is True
assert manifest["model_refitted_during_evaluation"] is False
assert manifest["model_reselected_during_evaluation"] is False
assert manifest["calibration_reselected_during_evaluation"] is False
assert manifest["event_probabilities_calculated"] is False
assert manifest["market_data_accessed"] is False
assert manifest["trading_returns_calculated"] is False

print(
    "Evaluation completed without changing the locked design:",
    True,
)

Evaluation completed without changing the locked design: True


## Evidential boundary

This notebook answers whether probabilistic post-processing improves
the raw deterministic temperature forecast.

It does not yet answer whether the forecast outperforms Polymarket or
generates economic value. Those questions require event probabilities,
market prices and a separately specified trading rule.